In [5]:

# ── MODEL ───────────────────────────────────────────────────────────────────────
print("Loading model (this may take a minute)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,   # CPU requires float32, not bfloat16
    device_map="cpu",
)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded: {total_params:.0f}M params")

# ── LORA ────────────────────────────────────────────────────────────────────────
print("Applying LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# expect something like: trainable params: 1.7M || all params: 600M (~0.28%)

# ── DATASET ─────────────────────────────────────────────────────────────────────
print("Loading dataset...")
dataset = load_dataset("json", data_files={"train": DATA_PATH}, split="train")
print(f"{len(dataset)} examples loaded")
print("First example:", dataset[0])   # sanity check your format

# ── TRAINER ─────────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,  # simulates batch size of 8
        warmup_steps=10,
        max_steps=MAX_STEPS,
        logging_steps=10,
        output_dir=OUTPUT_DIR,
        optim="adamw_torch",            # adamw_8bit is CUDA only
        save_steps=100,
        seed=3407,
        dataset_num_proc=1,
        fp16=False,                     # CPU can't do fp16 training
        bf16=False,
    ),
)

# ── TRAIN ───────────────────────────────────────────────────────────────────────
print("Starting training... (grab a coffee, this will take a while on CPU)")
trainer.train()
print("Training complete!")

# ── SAVE ────────────────────────────────────────────────────────────────────────
print(f"Saving LoRA adapter to {ADAPTER_DIR}...")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Done! Your adapter is saved.")
# the adapter is tiny (~10-30MB), not the full model
# to use it later: model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

Loading model (this may take a minute)...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model loaded: 376M params
Applying LoRA...
trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
Loading dataset...


FileNotFoundError: Unable to find '/home/saury/Projects/personal/gurGPT/your_data.jsonl'